---
title: Dataclass vs namedtuple in Python
created: '2020-10-14T23:34:04-07:00'
date: '2026-09-30T18:18:21-07:00'
authors:
  - bendu
label: dataclass-vs-namedtuple-in-python
license: CC-BY-4.0
tags:
  - computer science
  - Python
  - programming
  - collections
  - namedtuple
  - dataclass
---

**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


## Tips and Traps 

1. Prefer `Dataclass` to `namedtuple` for many reasons.
    - A namedtuple is immutable while a dataclass can be both 
        mutable (`frozen=False` which is the default) or immutable (`frozen=True`).
        
    However,
    namedtuple does have one advantage over dataclass. 
    Members of a namedtuple is assible both via the dot operator and index.
    In situations where both dot accessing and index accessing of members is required,
    a namedtuple comes handy.
    For examples, 
    a list of namedtuple objects can be used as the data for creating a pandas DataFrame
    but not a list of dataclass objects.


## dataclass - mutable

In [1]:
from dataclasses import dataclass


@dataclass
class Person:
    name: str
    age: int = 10

In [2]:
Person("Ben")

Person(name='Ben', age=10)

In [3]:
p = Person("Ben", 34)
p

Person(name='Ben', age=34)

In [4]:
str(p)

"Person(name='Ben', age=34)"

## dataclass - immutable

Attributes of the class `Person` is mutable
since `frozen=False` (default).

In [13]:
p.age = 20
p

Person(name='Ben', age=20)

In [18]:
@dataclass(frozen=True)
class PersonImmutable:
    name: str
    age: int = 10

In [19]:
PersonImmutable("Ben")

PersonImmutable(name='Ben', age=10)

In [20]:
p = PersonImmutable("Ben", 30)
p

PersonImmutable(name='Ben', age=30)

Attribute of `PersonImmutable` is immutable since `frozen=True`.

In [22]:
p.age = 20

FrozenInstanceError: cannot assign to field 'age'

## Dataclass Members with Default Values

In Python, fields of a dataclass instance are accessed via the dot operator (e.g., `obj.member`).
When a field is defined with a default value (e.g., `name: str = "Alice"`),
it also exists as a **class variable** (class attribute) on the dataclass itself.

Consequently, you can reference it directly via the class (`MyDataClass.name`)
as well as via an instance (`obj.name`).


In [23]:
from dataclasses import dataclass


@dataclass
class MyDataClass:
    name: str = "Alice"


# Accessing via the class directly
print(MyDataClass.name)

# Accessing via an instance
obj = MyDataClass()
print(obj.name)

Alice
Alice


### How It Works Under the Hood

When you define a class in Python, anything assigned directly within the class body becomes a **class attribute**.

1. When you write `name: str = "Alice"`, Python evaluates and stores `"Alice"` on the `MyDataClass` class object itself. (In contrast, fields declared without default values, such as `name: str`, only exist as type annotations in `__annotations__` and do not create a class attribute.)
2. The `@dataclass` decorator inspects the class annotations and attributes, automatically generating an `__init__` method based on them.
3. When you instantiate the object (`obj = MyDataClass()`), the generated `__init__` method assigns the default value to `self.name` on the instance, creating an **instance attribute** that shadows the class attribute. Because it is an instance attribute, modifying `obj.name` on an instance does not alter `MyDataClass.name`, and reassigning `MyDataClass.name` after creation does not affect existing instances.

> **Note:** If you want a true shared class variable that should *not* be treated as an instance field or included in `__init__`, annotate it with `typing.ClassVar` (e.g., `species: ClassVar[str] = "Human"`). Dataclasses intentionally ignore `ClassVar` attributes during code generation.


### The Exception: `default_factory` for Mutable Defaults

There is one major scenario where a default value is **not** available as a class variable: when using `default_factory`.

Because sharing mutable defaults (such as lists or dictionaries) across all instances of a class causes subtle bugs in Python, dataclasses forbid direct mutable defaults (e.g., `items: list = []` raises a `ValueError`) and require `field(default_factory=...)` instead.

When you use a factory, the default value is generated dynamically during instantiation inside `__init__`, so it is never assigned to the class itself. Attempting to access it directly via the class raises an `AttributeError`.


In [24]:
from dataclasses import dataclass, field


@dataclass
class MyDataClassWithList:
    items: list = field(default_factory=list)


# Accessing via an instance works
obj = MyDataClassWithList()
print(obj.items)

[]


In [25]:
# Accessing via the class directly raises an AttributeError
try:
    print(MyDataClassWithList.items)
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: type object 'MyDataClassWithList' has no attribute 'items'


## namedtuple

In [2]:
from collections import namedtuple

In [7]:
PersonNT = namedtuple("PersonNT", ["name", "age"])

In [8]:
p = PersonNT("Ben Du", 30)
p

PersonNT(name='Ben Du', age=30)

In [9]:
p[0]

'Ben Du'

In [10]:
p[1]

30

## Objects of namedtuple as pandas DataFrame Data

In [11]:
import pandas as pd

In [12]:
pd.DataFrame(data=[p])

,name,age
0,Ben Du,30


## References

https://docs.python.org/3/library/collections.html#collections.namedtuple

[Difference between DataClass vs NamedTuple vs Object in Python](https://www.geeksforgeeks.org/difference-between-dataclass-vs-namedtuple-vs-object-in-python/)

[DataClass vs NamedTuple vs Object: A Battle of Performance in Python](https://medium.com/@jacktator/dataclass-vs-namedtuple-vs-object-for-performance-optimization-in-python-691e234253b9)